# Quantum ESPRESSO — Environment Setup

This notebook installs and configures a **Quantum ESPRESSO 7.5** conda environment in Google Colab using [condacolab](https://github.com/conda-incubator/condacolab). The QE package on conda-forge is named `qe`.

Run it **once** before starting any QE tutorial notebook. Optionally, you can save the environment to Google Drive so future sessions restore in ~1 minute instead of ~10 minutes.

---

## How to use this notebook

| Step | Cell | What it does |
|------|------|--------------|
| 1 | Bootstrap condacolab | Installs condacolab and **restarts the kernel** (expected) |
| 2 | Build the QE environment | Creates `qe_env` with QE 7.5 (~10 min) |
| 3 | Verify the installation | Confirms `pw.x` is available |
| 4 *(optional)* | Save to Drive | Packs the env to Drive for fast restore later |

> ⚠️ **Important:** After Step 1 the kernel restarts automatically. When it does, **re-run all cells from the top** — the restart is expected and required by condacolab.

---
## Step 1 — Bootstrap condacolab

This cell installs condacolab and replaces Colab's default Python environment with a full Miniforge conda installation.

**What to expect:**
- You will see a *"Your session crashed for an unknown reason"* message — **this is normal**.
- The kernel restarts automatically.
- After the restart, run this cell again and it will skip the install and proceed to Step 2.

In [ ]:
# ── Step 1: Bootstrap condacolab ─────────────────────────────────────────────
try:
    import condacolab
    condacolab.check()          # already installed — skip straight to Step 2
    print("✅ condacolab already active. Continue to Step 2.")
except Exception:
    import subprocess, sys
    print("Installing condacolab …")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "condacolab"],
        stdout=subprocess.DEVNULL,
    )
    import condacolab
    condacolab.install()        # ← triggers kernel restart here

---
## Step 2 — Build the Quantum ESPRESSO environment

Creates a conda environment named `qe_env` with:
- `qe=7.5` (from conda-forge)
- `numpy`, `matplotlib` — for post-processing and plotting
- `ase` — Atomic Simulation Environment (useful for building input files)
- `ovito` — visualization and analysis of atomistic simulation data
- `conda-pack` — needed if you want to save the env to Drive (Step 4)

> ⏱️ **This takes approximately 5–10 minutes.** conda needs to resolve and download the full QE stack (MPI, LAPACK, FFTW, …). Grab a coffee.

In [ ]:
# ── Step 2: Create the qe_env conda environment ───────────────────────────────
import condacolab, subprocess
condacolab.check()

ENV_NAME   = "qe_env"
QE_VERSION = "7.5"
PY_VERSION = "3.11"

print(f"Creating environment '{ENV_NAME}' with quantum-espresso={QE_VERSION} …")
print("This may take 5–10 minutes.\n")

subprocess.run(
    [
        "conda", "create",
        "--name", ENV_NAME,
        "-c", "conda-forge",
        "--yes",
        f"python={PY_VERSION}",
        f"qe={QE_VERSION}",
        "numpy",
        "matplotlib",
        "ase",
        "ovito",                # visualization and analysis of atomistic data
        "conda-pack",           # needed for Drive save (Step 4)
    ],
    check=True,
)

print(f"\n✅ Environment '{ENV_NAME}' created successfully.")

---
## Step 3 — Activate and verify

Prepends `qe_env/bin` to `PATH` so all QE executables (`pw.x`, `ph.x`, …) are reachable from any cell or shell command.

In [ ]:
# ── Step 3: Add qe_env to PATH and verify ────────────────────────────────────
import subprocess, os

ENV_BIN = "/opt/conda/envs/qe_env/bin"

# Prepend the env bin dir to PATH so QE executables are found
os.environ["PATH"] = ENV_BIN + ":" + os.environ["PATH"]
print(f"✅ Added {ENV_BIN} to PATH")

# Check pw.x is reachable
result = subprocess.run(["pw.x", "--version"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"🎉 pw.x found: {result.stdout.strip()}")
else:
    print("⚠️  pw.x not found — double-check ENV_BIN path above.")
    print(result.stderr)

# List key QE executables
execs = ["pw.x", "ph.x", "pp.x", "bands.x", "dos.x"]
print("\nQE executables on PATH:")
for exe in execs:
    found = subprocess.run(["which", exe], capture_output=True, text=True)
    status = "✅" if found.returncode == 0 else "❌"
    print(f"  {status}  {exe:12s}  {found.stdout.strip()}")

print("\nSetup complete — you can now run QE tutorial notebooks.")

---
## Step 4 (Optional) — Save the environment to Google Drive

Packing the environment to Drive lets you **restore it in ~1 minute** in future sessions instead of rebuilding from scratch.

### Requirements
- A Google account with **~2–3 GB of free Drive space**
- You only need to do this **once** (or when you want to update the env)

### Sharing with others
If you are an instructor or want to share the env with colleagues:
1. Run the save cell below
2. Share the folder `My Drive/conda_envs/` with your collaborators (view-only is enough)
3. Give them the path to use in the restore notebook

> ⏱️ Packing takes 3–5 minutes. The resulting `.tar.gz` is approximately 2–3 GB.

In [ ]:
# ── Step 4a: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")
print("✅ Drive mounted at /content/drive")

In [ ]:
# ── Step 4b: Pack and save the environment ────────────────────────────────────
import subprocess, os

SAVE_DIR  = "/content/drive/MyDrive/conda_envs"
SAVE_PATH = f"{SAVE_DIR}/qe_env.tar.gz"

os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Packing 'qe_env' → {SAVE_PATH}")
print("This takes 3–5 minutes …\n")

subprocess.run(
    [
        "conda", "run", "-n", "qe_env",
        "conda-pack",
        "-n", "qe_env",
        "-o", SAVE_PATH,
        "--ignore-editable-packages",
        "--force",              # overwrite if file already exists
    ],
    check=True,
)

size_gb = os.path.getsize(SAVE_PATH) / 1e9
print(f"\n✅ Environment saved to Drive ({size_gb:.1f} GB)")
print(f"   Path: {SAVE_PATH}")
print("\nShare this path with collaborators so they can restore the env quickly.")

---
## What to put at the top of every tutorial notebook

Once this setup notebook has been run (and optionally saved to Drive), paste **one** of the following cells at the top of each tutorial notebook.

### Option A — Rebuild each session (no Drive required)

```python
# ── QE environment bootstrap ─────────────────────────────
try:
    import condacolab; condacolab.check()
except Exception:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "condacolab"])
    import condacolab; condacolab.install()   # restarts kernel

# ── after restart: add qe_env to PATH ────────────────────
import os
os.environ["PATH"] = "/opt/conda/envs/qe_env/bin:" + os.environ["PATH"]
print("✅ qe_env active")
```

### Option B — Restore from Google Drive (~1 min, recommended)

```python
# ── QE environment bootstrap (Drive restore) ─────────────
try:
    import condacolab; condacolab.check()
except Exception:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "condacolab"])
    import condacolab; condacolab.install()   # restarts kernel

# ── after restart: restore from Drive and activate ───────
import condacolab, subprocess, os
from google.colab import drive

condacolab.check()
drive.mount("/content/drive")

ENV_ARCHIVE = "/content/drive/MyDrive/conda_envs/qe_env.tar.gz"
ENV_PATH    = "/opt/conda/envs/qe_env"

if not os.path.isdir(ENV_PATH):
    print("Restoring qe_env from Drive …")
    os.makedirs(ENV_PATH, exist_ok=True)
    subprocess.run(["tar", "-xzf", ENV_ARCHIVE, "-C", ENV_PATH], check=True)
    print("✅ Environment restored.")
else:
    print("✅ qe_env already present.")

os.environ["PATH"] = "/opt/conda/envs/qe_env/bin:" + os.environ["PATH"]
print("✅ qe_env active")
```

> **Tip for instructors:** Replace `MyDrive/conda_envs/` with your shared folder path and distribute Option B to all participants — they restore your pre-built env with no storage cost on their end.